# Vision Hallucination Metrics

Evaluate a Vision Language Model (VLM) for hallucination in detection systems like Argos.

Classification is derived automatically from two boolean labels per event:
- `agentic["detected"]` — what the VLM reported
- `ground_truth_agentic["detected"]` — the human ground truth label

| Metric | Formula | What it tells you |
|--------|---------|-------------------|
| **FalsePositiveRate** | FP / (FP + TN) | How often the model invents events that never occurred. A high FPR floods operators with false alarms. |
| **Precision** | TP / (TP + FP) | Of all the alerts raised, how many were real. Low precision means operators start ignoring alerts. |
| **ConfidenceScoreAnalysis** | ECE + distribution stats | Whether confidence scores are reliable for threshold tuning. ECE measures the gap between how confident the model says it is and how often it's actually right. |

In [ ]:
!pip install alquimia-fair-forge

In [ ]:
import logging
from fair_forge.metrics.vision import ConfidenceScoreAnalysis, FalsePositiveRate, Precision

logging.getLogger('httpx').disabled = True

## Setup

Implementá un `Retriever` que cargue tu dataset de evaluación de Argos.

Cada `Batch` debe tener:
- `assistant` — descripción libre del VLM sobre el frame
- `ground_truth_assistant` — descripción humana de lo que realmente ocurrió
- `ground_truth_agentic["detected"]` — `True/False`: ¿hubo realmente un evento?
- `agentic["confidence"]` — score de confianza del modelo (opcional, necesario para `ConfidenceScoreAnalysis`)

La clasificación TP/FP/TN/FN se deriva automáticamente comparando las dos descripciones con similitud semántica. El `threshold` (por defecto `0.75`) es configurable — ajustalo según tu dominio.

In [ ]:
import json
from pathlib import Path
from fair_forge import Retriever
from fair_forge.schemas import Dataset


class VLMRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        with open(Path('dataset.json'), encoding='utf-8') as f:
            return [Dataset.model_validate(item) for item in json.load(f)]


# dataset.json — estructura de cada evento:
#
# {
#   "session_id": "cam-lobby-01",
#   "assistant_id": "argos-gpt4o",
#   "language": "english",
#   "context": "Lobby entrance camera",
#   "conversation": [
#     {
#       "qa_id": "2026-03-13T14:00:00Z",
#       "query": "Describe what you observe in this frame.",
#       "assistant": "A person is lying on the floor near the entrance.",
#       "ground_truth_assistant": "A person fell near the entrance.",
#       "agentic": { "confidence": 0.91 },
#       "ground_truth_agentic": { "detected": true }
#     }
#   ]
# }

## False Positive Rate

How often does the model invent events that never occurred?

**FPR = FP / (FP + TN)**

In [ ]:
fpr_results = FalsePositiveRate.run(VLMRetriever, threshold=0.75)

for r in fpr_results:
    fpr = r.false_positive_rate
    print(f'{r.session_id}')
    print(f'  FPR          : {fpr:.1%}' if fpr is not None else '  FPR          : N/A (no actual negatives)')
    print(f'  False alarms : {r.n_false_positives} / {r.n_negatives} real negatives')
    print(f'  Total events : {r.n_predictions}')
    print()

## Precision

When the model raised an alert, how often was it correct?

**Precision = TP / (TP + FP)**

In [ ]:
precision_results = Precision.run(VLMRetriever, threshold=0.75)

for r in precision_results:
    prec = r.precision
    print(f'{r.session_id}')
    print(f'  Precision    : {prec:.1%}' if prec is not None else '  Precision    : N/A (no positive predictions)')
    print(f'  True alerts  : {r.n_true_positives} / {r.n_positive_predictions} alerts raised')
    print(f'  Total events : {r.n_predictions}')
    print()

## Confidence Score Analysis

Are the VLM's confidence scores calibrated? A score of 0.8 should mean the model is right ~80% of the time.

Reads `Batch.agentic["confidence"]` and computes distribution stats + Expected Calibration Error (ECE).

In [ ]:
confidence_results = ConfidenceScoreAnalysis.run(VLMRetriever, threshold=0.75)

for r in confidence_results:
    print(f'{r.session_id}')
    print(f'  Events with confidence : {r.n_with_confidence} / {r.n_predictions}')
    if r.confidence_mean is not None:
        print(f'  Confidence             : mean={r.confidence_mean:.2f}  std={r.confidence_std:.2f}  min={r.confidence_min:.2f}  max={r.confidence_max:.2f}')
    ece = r.expected_calibration_error
    print(f'  ECE                    : {ece:.3f}' if ece is not None else '  ECE                    : N/A')
    print()
    for b in [b for b in r.buckets if b.count > 0]:
        conf = f'{b.mean_confidence:.0%}'
        acc = f'{b.accuracy:.0%}'
        print(f'  [{b.range_min:.1f}–{b.range_max:.1f}]  n={b.count}  conf={conf}  acc={acc}')
    print()